In [ ]:
!pip install git+https://github.com/cvg/LightGlue.git
!pip install torch torchvision numpy opencv-python matplotlib scipy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import matplotlib.pyplot as plt
from lightglue import SuperPoint
from scipy.spatial.distance import cdist
from google.colab.patches import cv2_imshow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Environnement prêt sur : {device}")

  Cloning https://github.com/cvg/LightGlue.git to /tmp/pip-req-build-0gipy329
  Running command git clone --filter=blob:none --quiet https://github.com/cvg/LightGlue.git /tmp/pip-req-build-0gipy329
  Resolved https://github.com/cvg/LightGlue.git to commit 746fac2c042e05d1865315b1413419f1c1e7ba55
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 109.4 MB/s eta 0:00:00
  Created wheel for lightglue: filename=lightglue-0.0-py3-none-any.whl size=40023 sha256=636ba066d56e99020d4f635362db9408d35802dc3d071ca3c2c5c743b359f012
  Stored in directory: /tmp/pip-ephem-wheel-cache-1oxhhttj/wheels/dc/16/88/ad4ddb490c3a6ff37eeface44b776cb588b19e169210e0fbf1
Successfully built lightglue
✅ Environnement prêt sur : cuda


In [ ]:
class SuperPointPruned(nn.Module):
    def __init__(self, scale=0.75): # On garde 75% des canaux
        super().__init__()
        c1, c2, c3, c4 = int(64*scale), int(64*scale), int(128*scale), int(128*scale)

        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv1a = nn.Conv2d(1, c1, kernel_size=3, stride=1, padding=1)
        self.conv1b = nn.Conv2d(c1, c1, kernel_size=3, stride=1, padding=1)
        self.conv2a = nn.Conv2d(c1, c2, kernel_size=3, stride=1, padding=1)
        self.conv2b = nn.Conv2d(c2, c2, kernel_size=3, stride=1, padding=1)
        self.conv3a = nn.Conv2d(c2, c3, kernel_size=3, stride=1, padding=1)
        self.conv3b = nn.Conv2d(c3, c3, kernel_size=3, stride=1, padding=1)
        self.conv4a = nn.Conv2d(c3, c4, kernel_size=3, stride=1, padding=1)
        self.conv4b = nn.Conv2d(c4, c4, kernel_size=3, stride=1, padding=1)

        self.convPa = nn.Conv2d(c4, 256, kernel_size=3, stride=1, padding=1)
        self.convPb = nn.Conv2d(256, 65, kernel_size=1, stride=1, padding=0)
        self.convDa = nn.Conv2d(c4, 256, kernel_size=3, stride=1, padding=1)
        self.convDb = nn.Conv2d(256, 256, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x = self.relu(self.conv1a(x)); x = self.relu(self.conv1b(x)); x = self.pool(x)
        x = self.relu(self.conv2a(x)); x = self.relu(self.conv2b(x)); x = self.pool(x)
        x = self.relu(self.conv3a(x)); x = self.relu(self.conv3b(x)); x = self.pool(x)
        x = self.relu(self.conv4a(x)); feats = self.relu(self.conv4b(x))

        heatmap = self.convPb(self.relu(self.convPa(feats)))
        desc = self.convDb(self.relu(self.convDa(feats)))
        desc = F.normalize(desc, p=2, dim=1)
        return heatmap, desc

# Helper pour le Teacher (LightGlue structure)
def get_teacher_outputs(m, img):
    x = F.relu(m.conv1a(img)); x = F.relu(m.conv1b(x)); x = F.max_pool2d(x, 2)
    x = F.relu(m.conv2a(x)); x = F.relu(m.conv2b(x)); x = F.max_pool2d(x, 2)
    x = F.relu(m.conv3a(x)); x = F.relu(m.conv3b(x)); x = F.max_pool2d(x, 2)
    x = F.relu(m.conv4a(x)); f = F.relu(m.conv4b(x))
    h = m.convPb(m.relu(m.convPa(f)))
    d = m.convDb(m.relu(m.convDa(f)))
    return h, F.normalize(d, p=2, dim=1)

# Instanciation
teacher = SuperPoint(max_num_keypoints=None).eval().to(device)
student = SuperPointPruned(scale=0.75).to(device)

Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth" to /root/.cache/torch/hub/checkpoints/superpoint_v1.pth


100%|██████████| 4.96M/4.96M [00:00<00:00, 67.5MB/s]


In [ ]:
class SyntheticShapes(Dataset):
    def __init__(self, size=1500): self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx):
        img = np.zeros((240, 320), dtype=np.uint8)
        for _ in range(np.random.randint(4, 10)):
            p1, p2 = (np.random.randint(0, 320), np.random.randint(0, 240)), (np.random.randint(0, 320), np.random.randint(0, 240))
            if np.random.rand() > 0.5: cv2.circle(img, p1, np.random.randint(10, 40), 255, -1)
            else: cv2.rectangle(img, p1, p2, 255, -1)
        return torch.from_numpy(img).float().unsqueeze(0) / 255.0

dataloader = DataLoader(SyntheticShapes(), batch_size=32, shuffle=True)
optimizer = optim.Adam(student.parameters(), lr=0.0004)

print(" Distillation en cours (Scale 0.75)...")
for epoch in range(15):
    student.train(); epoch_loss = 0
    for batch in dataloader:
        batch = batch.to(device)
        with torch.no_grad(): t_h, t_d = get_teacher_outputs(teacher, batch)
        s_h, s_d = student(batch)

        # Loss Hybride (CE + Cosine)
        loss_det = torch.mean(torch.sum(-F.softmax(t_h, dim=1) * F.log_softmax(s_h, dim=1), dim=1))
        loss_desc = 1 - F.cosine_similarity(s_d, t_d, dim=1).mean()
        loss = loss_det + 20 * loss_desc

        optimizer.zero_grad(); loss.backward(); optimizer.step(); epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/15 | Loss: {epoch_loss/len(dataloader):.4f}")

torch.save(student.state_dict(), "sp_pruned_075.pth")

🚀 Distillation en cours (Scale 0.75)...
Epoch 1/15 | Loss: 14.6635
Epoch 2/15 | Loss: 11.6281
Epoch 3/15 | Loss: 9.3669
Epoch 4/15 | Loss: 5.9752
Epoch 5/15 | Loss: 4.3254
Epoch 6/15 | Loss: 3.3878
Epoch 7/15 | Loss: 2.7693
Epoch 8/15 | Loss: 2.2893
Epoch 9/15 | Loss: 1.9563
Epoch 10/15 | Loss: 1.7161
Epoch 11/15 | Loss: 1.5544
Epoch 12/15 | Loss: 1.4076
Epoch 13/15 | Loss: 1.3253
Epoch 14/15 | Loss: 1.2535
Epoch 15/15 | Loss: 1.1819


In [ ]:
def benchmark_precision(pts_ref, pts_test, eps=3):
    if len(pts_ref) == 0 or len(pts_test) == 0: return 0.0, 0.0
    dists = cdist(pts_ref, pts_test); min_dist = np.min(dists, axis=1)
    matches = min_dist <= eps
    return (np.sum(matches)/len(pts_ref)*100), (np.mean(min_dist[matches]) if np.any(matches) else 0.0)

# 1. Test de Répétabilité
student.eval()
img_test = np.zeros((240, 320), dtype=np.uint8)
cv2.rectangle(img_test, (40, 40), (120, 120), 255, 2); cv2.circle(img_test, (200, 150), 40, 255, 2)
input_t = torch.from_numpy(img_test).float()[None, None, :, :].to(device) / 255.0

with torch.no_grad():
    t_h, _ = get_teacher_outputs(teacher, input_t)
    s_h, _ = student(input_t)

def decode(h, t=0.015):
    p = torch.softmax(h, dim=1)[:, :-1].permute(0, 2, 3, 1).reshape(30, 40, 8, 8).permute(0, 2, 1, 3).reshape(240, 320).cpu().numpy()
    idx = np.where(p > t); return np.stack([idx[1], idx[0]], axis=-1)

pts_t = decode(t_h)
pts_s = decode(s_h, t=0.005) # Seuil légèrement plus bas pour le student
rep, mle = benchmark_precision(pts_t, pts_s)

print(f"📊 BILAN FINAL (Scale 0.75) : Répétabilité {rep:.2f}% | MLE {mle:.4f}px")

# 2. CONVERSION FP16
student_fp16 = student.half()
torch.save(student_fp16.state_dict(), "superpoint_v075_fp16.pth")
print("✨ Modèle exporté en FP16 !")

📊 BILAN FINAL (Scale 0.75) : Répétabilité 53.63% | MLE 0.8957px
✨ Modèle exporté en FP16 !
